In [17]:
from rich.markdown import Markdown
from rich import print
import threading
from collections.abc import Sequence
from typing import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from langgraph.graph import (
    START,
    END,
    StateGraph,
    add_messages
)
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.state import RunnableConfig

In [18]:
# definindo o llm utilizado
llm = init_chat_model("ollama:gemma:2b")

In [19]:
# 1º defino meu state
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# 2º defino meu nodes
def call_llm(state: AgentState) -> AgentState:
    llm_result = llm.invoke(state["messages"])
    return {"messages": [llm_result]}

# 3º crio o StateGraph
builder = StateGraph(
    AgentState,
    context_schema=None,
    input_schema=AgentState,
    output_schema=AgentState
)

# 4º adicionando nodes ao grafo
builder.add_node("call_llm", call_llm)
builder.add_edge(START, "call_llm")
builder.add_edge("call_llm", END)

# 5º compilar o grafo
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
config = RunnableConfig(configurable={"thread_id": threading.get_ident()})

In [20]:
print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| call_llm |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+

In [21]:
while True:
    user_input = input("Digite sua mensage: ")
    print(Markdown("---"))

    if user_input.lower() in ["q", "quit"]:
        print("Bye 👋")
        print(Markdown("---"))
        break

    human_message = HumanMessage(user_input)
    result = graph.invoke({"messages": [human_message]}, config=config)

    print(Markdown(str(result["messages"][-1].content)))
    print(Markdown("---"))

Digite sua mensage:  olá


───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Olá! Como posso ajudar você hoje?

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Digite sua mensage:  como você vai?


...

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Digite sua mensage:  q


───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Bye 👋

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────